# Imports

In [ ]:
import pandas as pd
from pathlib import Path

# The Strictest Score

1 point for each interaction from the crystal structure that is present in the docked pose. No points for anything else.

In [ ]:
# Load the data

In [ ]:
ints_path = Path("interactions_analysis")

In [ ]:
xtal1 = pd.read_csv(ints_path / "SARS_A0181_ASAP-0000153-001_refine_min_interactions.csv", index_col=0)

In [ ]:
docked1 = pd.read_csv(ints_path / "ASAP-0000153_min_interactions.csv", index_col=0)

In [ ]:
merged = pd.concat([xtal1,docked1])

In [ ]:
sum(merged.groupby(merged.columns.tolist(),as_index=False).size()['size'] > 1)

In [ ]:
xtal2 = pd.read_csv(ints_path / "SARS_A0221_ASAP-0000526-001_refine_min_interactions.csv", index_col=0)
docked2 = pd.read_csv(ints_path / "ASAP-0000526_min_interactions.csv", index_col=0)

In [ ]:
merged2 = pd.concat([xtal2,docked2])

In [ ]:
xtal2

In [ ]:
merged2.groupby(merged2.columns.tolist(),as_index=False).size()

In [ ]:
xtal3 = pd.read_csv(ints_path / "SARS_x0589_ASAP-0008314-001_refine_min_interactions.csv", index_col=0)
docked3 = pd.read_csv(ints_path / "ASAP-0008314_min_interactions.csv", index_col=0)

In [ ]:
merged3 = pd.concat([xtal3,docked3])

In [ ]:
sum(merged3.groupby(merged3.columns.tolist(),as_index=False).size()['size'] > 1)

In [ ]:
merged3.groupby(merged3.columns.tolist(),as_index=False).size()

In [ ]:
from enum import Enum
class InteractionScoreLevel(Enum):
    STRICT = 1 # 1 point for each interaction that is exactly the same, 0 points for anything else
    BY_TYPE = 2 # 1 point for each interaction that has the same type as an interaction in the Crystal Structure
    VERY_LOOSE = 3


from pydantic import BaseModel
import abc

class InteractionScore(BaseModel):
    provenance: str
    found_interactions: int
    xtal_interactions: int
    matched_interactions: int
    union_size: int
    normalized_score: float
    tanimoto_coefficient: float
    tversky_index: float

class InteractionScorer(BaseModel):
    
    name: str = "Interaction Score Base"
    
    @abc.abstractmethod
    def _compute_score(self, xtal, docked):
        pass
    
    def compute_score(self, xtal, docked) -> InteractionScore:
        """
        This function returns a single score that describes how well a docked pose reproduces the interactions seen in the crystal structure.
        
        :param xtal: 
        :param docked: 
        :return: 
        """
        return self._compute_score(xtal, docked)

class StrictInteractionScorer(InteractionScorer):
    name = "Strict Interaction Score"
    description = ""
    
    def _compute_score(self, xtal, docked):
        merged = pd.concat([xtal,docked])
        union_df = merged.groupby(merged.columns.tolist(),as_index=False).size()
        union = len(union_df)
        raw_score = sum(union_df["size"] > 1)
        normalized = raw_score / len(xtal)
        return InteractionScore(provenance=self.name, matched_interactions=raw_score, normalized_score=normalized, union_size=union, xtal_interactions=len(xtal), found_interactions=len(docked))

class ByTypeInteractionScorer(InteractionScorer):
    name = "By Type Interaction Score"
    description = "1 point for each interaction that has the same type as an interaction in the Crystal Structure"
    
    def _compute_score(self, xtal, docked):
        pass
        
    

In [ ]:
score = StrictInteractionScorer()

In [ ]:
int_score = score.compute_score(xtal1, docked1)
print(int_score.tanimoto_coefficient)
print(int_score.tversky_index)

In [ ]:
int_score = score.compute_score(xtal2, docked2)
print(int_score.tanimoto_coefficient)
print(int_score.tversky_index)

In [ ]:
int_score = score.compute_score(xtal3, docked3)
print(int_score.tanimoto_coefficient)
print(int_score.tversky_index)

# Version 2

I'd like to do this differently where the pipeline looks more like:
1. Collect PLINT reports
2. Convert to Fingerprints
3. Compare Fingerprints using various scores

In [ ]:
import plip_analysis as pa
from importlib import reload
reload(pa)

In [ ]:
plint_docked = pa.PLIntReport.from_csv(ints_path / "SARS_A0181_ASAP-0000153-001_refine_min_interactions.csv")
plint_xtal = pa.PLIntReport.from_csv(ints_path / "ASAP-0000153_min_interactions.csv")

In [ ]:
plint_xtal

In [ ]:
int_types = [a.value for a in pa.InteractionType]

In [ ]:
from enum import Enum, auto
from plip_analysis import PLIntReport
class FingerprintLevel(Enum):
    ByInteractionType = 'ByInteractionType'
    ByInteractionTypeAndResidueType = 'ByInteractionTypeAndResidueType'
    ByInteractionTypeAndAtomTypes = 'ByInteractionTypeAndAtomTypes'
    ByInteractionTypeAndResidueTypeAndBBorSC = 'ByInteractionTypeAndResidueTypeAndBBorSC'
    ByInteractionTypeAndResidueTypeAndNumber = 'ByInteractionTypeAndResidueTypeAndNumber'
    ByEverything = 'ByEverything'
    
    def __str__(self):
        return self.value
    

def calculate_fingerprint(plint_report: PLIntReport, level: FingerprintLevel) -> dict:
    fingerprint_dict = {}
    for interaction in plint_report.interactions:
        if level == FingerprintLevel.ByInteractionType:
            key = interaction.interaction_type.value
        elif level == FingerprintLevel.ByInteractionTypeAndResidueType:
            key = f"{interaction.interaction_type.value}_{interaction.protein_residue_type}"
        elif level == FingerprintLevel.ByInteractionTypeAndAtomTypes:
            key = f"{interaction.interaction_type.value}_Protein_{interaction.protein_atom_type}_Ligand_{interaction.ligand_atom_type}"
        elif level == FingerprintLevel.ByInteractionTypeAndResidueTypeAndBBorSC:
            key = f"{interaction.interaction_type.value}_{interaction.protein_residue_type}_{'SC' if interaction.to_sidechain else 'BB'}"
        elif level == FingerprintLevel.ByInteractionTypeAndResidueTypeAndNumber:
            key = f"{interaction.interaction_type.value}_{interaction.protein_residue_type}{interaction.protein_residue_number}"
        elif level == FingerprintLevel.ByEverything:
            key = "_".join([f"{k}_{str(v)}" for k,v in interaction.dict().items()])
        else:
            raise ValueError("Invalid Fingerprint Level")
        original_count = fingerprint_dict.get(key, 0)
        fingerprint_dict[key] = original_count + interaction.count
    return fingerprint_dict

In [ ]:
class SimilarityScore(BaseModel):
    provenance: str
    score: float
    number_of_interactions_in_reference: int
    number_of_interactions_in_query: int
    number_of_interactions_in_intersection: int
    number_of_interactions_in_union: int
def calculate_tversky(fingerprint1:dict, fingerprint2:dict, alpha:float=1, beta:float=0):
    """
    Calculate the Tversky Index between two fingerprints.
    To calculate the Tanimoto Coefficient, set alpha=beta=1
    
    :param fingerprint1: 
    :param fingerprint2: 
    :param alpha: 
    :param beta: 
    :return: 
    """
    
    fp_types = set(fingerprint1.keys()).union(set(fingerprint2.keys()))
    matched = sum([min(fingerprint1.get(a,0), fingerprint2.get(a,0)) for a in fp_types])
    union = sum(fingerprint1.values()) + sum(fingerprint2.values()) - 2*matched
    score = matched / (matched + alpha * (sum(fingerprint1.values()) - matched) + beta * (sum(fingerprint2.values()) - matched))
    return SimilarityScore(provenance=f"Tversky_Alpha{alpha}_Beta{beta}", score=score, number_of_interactions_in_reference=sum(fingerprint1.values()), number_of_interactions_in_query=sum(fingerprint2.values()), number_of_interactions_in_intersection=matched, number_of_interactions_in_union=union,)

In [ ]:
class InteractionScore(BaseModel):
    provenance: str
    number_of_interactions_in_query: int
    number_of_interactions_in_reference: int
    number_of_interactions_in_intersection: int
    number_of_interactions_in_union: int
    tanimoto_coefficient: float
    tversky_index: float
    
    @classmethod
    def from_fingerprints(cls, reference:PLIntReport, query:PLIntReport, level:FingerprintLevel) -> "InteractionScore":
        reference = calculate_fingerprint(reference, level)
        query = calculate_fingerprint(query, level)
        tanimoto = calculate_tversky(reference, query, alpha=1, beta=1)
        tversky_index = calculate_tversky(reference, query)
        return cls(provenance=level.value, number_of_interactions_in_query=tanimoto.number_of_interactions_in_query, 
                   number_of_interactions_in_reference=tanimoto.number_of_interactions_in_reference,
                number_of_interactions_in_intersection=tanimoto.number_of_interactions_in_intersection,
                number_of_interactions_in_union=tanimoto.number_of_interactions_in_union,
                tanimoto_coefficient=tanimoto.score,
                tversky_index=tversky_index.score)
        

In [ ]:
score_list = []
for level in FingerprintLevel:
    score_list.append(InteractionScore.from_fingerprints(plint_xtal, plint_docked, level))

In [ ]:
int_types = [score.provenance for score in score_list]
tc = [score.tanimoto_coefficient for score in score_list]
tv = [score.tversky_index for score in score_list]

In [ ]:
df = pd.DataFrame({"Fingerprint Level": int_types, "Tversky Index": tv, "Tanimoto Coefficient": tc})

In [ ]:
df